# 01 · Data and ToT baseline

**Mini-crossword solver** (Avery Mattoon, Yutong Hu). CS 557 final project.

This notebook:
1. loads the ToT mini-crossword dataset and our fixed splits,
2. reproduces **ToT + GPT-4** from the trajectories the ToT authors released (free, no API calls) and checks our metric code against the paper,
3. re-runs the **ToT DFS baseline** with a model we choose, logging LLM calls, tokens and cost per puzzle,
4. saves results to `results/` so the other teammate can use them without re-running.

All real code lives in `src/crossword/`. Keep this notebook thin: if you change how something works, change it in `src/` so both of us get it.

**Runs on Colab, free.** In the 🔑 Secrets panel (left bar), add these with "Notebook access" on:
* `GITHUB_TOKEN`: a classic GitHub token with the `repo` scope (the repo is private)
* `GEMINI_API_KEY`: free key from [Google AI Studio](https://aistudio.google.com/apikey), only needed for section 3
* `OPENAI_API_KEY`: optional and paid; not needed

## 0 · Setup

In [15]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YutongHu-Natalie/mini_crossword_solver.git"
IN_COLAB = "google.colab" in sys.modules

def secret(name):
    """Colab secret, else environment variable / .env file."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

if IN_COLAB:
    # Clone the repo the first time; afterwards pull, so you always run the latest code.
    token = secret("GITHUB_TOKEN")
    print("GitHub token found:", bool(token))
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    if Path("/content/repo/.git").exists():
        # use the CURRENT token (an old clone may have a stale one saved)
        subprocess.run(["git", "-C", "/content/repo", "remote", "set-url", "origin", url])
        cmd = ["git", "-C", "/content/repo", "pull", "-q"]
    else:
        subprocess.run(["rm", "-rf", "/content/repo"])
        cmd = ["git", "clone", "-q", url, "/content/repo"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError(f"git failed:\n{msg}\nCheck the GITHUB_TOKEN secret (classic token, 'repo' "
                           "scope, notebook access on) and that you accepted the repo invite.")
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=True)
else:
    # running locally from notebooks/: work from the repo root
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    try:
        from dotenv import load_dotenv; load_dotenv()   # reads OPENAI_API_KEY etc. from .env
    except ImportError:
        pass

sys.path.insert(0, str(Path("src").resolve()))
for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY", "OPENAI_API_KEY"):
    if secret(name):
        os.environ[name] = secret(name)
if not os.environ.get("GEMINI_API_KEY") and os.environ.get("GOOGLE_API_KEY"):
    os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]   # Colab's "import key" button uses this name

# Optional: keep the LLM cache on Google Drive so it survives Colab restarts.
# from google.colab import drive; drive.mount("/content/drive")
# os.environ["CROSSWORD_CACHE"] = "/content/drive/MyDrive/crossword/llm_cache.sqlite"

print("repo root:", Path.cwd())
print("Gemini key found:", bool(os.environ.get("GEMINI_API_KEY")),
      "| OpenAI key found:", bool(os.environ.get("OPENAI_API_KEY")))

GitHub token found: True
repo root: /content/repo
Gemini key found: True | OpenAI key found: False


## 1 · Data and splits

In [16]:
import pandas as pd
from crossword import load_puzzles, get_split, TEST_IDS, FEWSHOT_IDS, DEV_IDS

puzzles = load_puzzles()
pd.DataFrame({
    "split": ["test", "fewshot", "dev"],
    "puzzles": [len(TEST_IDS), len(FEWSHOT_IDS), len(DEV_IDS)],
    "use": ["final numbers only (same 20 as ToT)",
            "worked examples inside ToT's prompts: never tune or test on these",
            "tuning k, prompts, search settings"],
})

,split,puzzles,use
0,test,20,final numbers only (same 20 as ToT)
1,fewshot,5,worked examples inside ToT's prompts: never tu...
2,dev,131,"tuning k, prompts, search settings"


In [17]:
print(puzzles[TEST_IDS[0]].render())

Puzzle 0
A G E N D
M O T O R
A R T S Y
S A L L E
S L E E R

h1. An agendum; something to be done
h2. An engine
h3. Pretentious; flowery
h4. A salon; a hall
h5. To mock; to sneer
v1. To heap
v2. An Indian antelope
v3. To intend; to plan; to devise; a nettle; to guess
v4. A nozzle
v5. Desiccator; more dry


Sanity check on the metric: the gold grid should score 100% and an empty grid 0%.

In [18]:
from crossword import score_board
p = puzzles[0]
print("gold :", score_board(list(p.solution), p.solution))
print("empty:", score_board(["_"] * 25, p.solution))

gold : {'letter': 1.0, 'word': 1.0, 'game': 1.0}
empty: {'letter': 0.0, 'word': 0.0, 'game': 0.0}


## 2 · ToT + GPT-4, from the paper's released trajectories

The ToT authors released every DFS trajectory from the paper (`data/tot_logs/`). Replaying them with our code should give the paper's Table 3: **ToT 78.0 / 60.0 / 20 %** (letter / word / game), **+best state 82.4 / 67.5 / 35 %**, **−prune 65.4 / 41.5 / 5 %**.

The paper's output rule is the *first deepest* state the DFS reached. "+best state" picks the state with the best word accuracy, which uses the answer key, so it is only an oracle upper bound.

The logs have no token counts, so below we rebuild every GPT-4 call that run needed from the trajectories (each one verified against ToT's released caches) and count the tokens. This gives the baseline's cost for free.

In [19]:
from crossword.experiment import tot_paper_records, summarize

paper = pd.DataFrame([
    {"method": "ToT + GPT-4 (paper logs)", **summarize(tot_paper_records(prune=True))},
    {"method": "ToT + GPT-4, no pruning (paper logs)", **summarize(tot_paper_records(prune=False))},
]).set_index("method")
paper

,n,letter_%,word_%,game_%,games_solved,oracle_letter_%,oracle_word_%,oracle_game_%,oracle_games_solved
method,,,,,,,,,
ToT + GPT-4 (paper logs),20,78.0,60.0,20.0,4,82.4,67.5,35.0,7
"ToT + GPT-4, no pruning (paper logs)",20,65.4,41.5,5.0,1,71.6,51.5,20.0,4


In [20]:
per_puzzle = pd.DataFrame([{"idx": r["idx"], **r["scores"], "nodes": r["nodes"]}
                           for r in tot_paper_records(prune=True)]).set_index("idx")
per_puzzle.T.round(2)

idx,0,5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95
letter,0.28,1.0,0.76,0.96,0.88,0.76,0.96,0.96,1.0,0.84,0.48,0.88,1.0,1.0,0.28,0.88,0.04,0.72,0.96,0.96
word,0.10,1.0,0.50,0.80,0.60,0.30,0.80,0.80,1.0,0.60,0.20,0.60,1.0,1.0,0.10,0.60,0.00,0.40,0.80,0.80
game,0.00,1.0,0.00,0.00,0.00,0.00,0.00,0.00,1.0,0.00,0.00,0.00,1.0,1.0,0.00,0.00,0.00,0.00,0.00,0.00
nodes,100.00,100.0,100.00,100.00,100.00,100.00,100.00,100.00,100.0,100.00,63.00,100.00,100.0,100.0,48.00,100.00,45.00,100.00,100.00,100.00


**What ToT + GPT-4 cost.** Input tokens are exact. ToT saved only the parsed scores from its 8 proposal samples, not the raw text, so output tokens are a lower bound. Dollars use GPT-4's list price ($30 / $60 per 1M input / output tokens).

In [21]:
from crossword.experiment import tot_paper_usage, count_tokens_gpt4

counter = count_tokens_gpt4()
print("token counts:", counter[1])
tot_cost = pd.DataFrame(tot_paper_usage(counter)).T
tot_cost.loc["mean per puzzle"] = tot_cost.mean().round(1)
tot_cost.loc["mean per puzzle", "usd (GPT-4)"] = round(
    (tot_cost.loc["mean per puzzle", "prompt_tokens"] * 30 + tot_cost.loc["mean per puzzle", "completion_tokens"] * 60) / 1e6, 2)
tot_cost.tail(1)

token counts: exact (tiktoken, gpt-4)


,calls,propose_calls,value_calls,prompt_tokens,completion_tokens,usd (GPT-4)
mean per puzzle,166.4,24.8,141.6,108192.6,17050.6,4.27


## 3 · (Optional) Re-run the ToT DFS baseline with our model

**You usually do not need this section.** Section 2 already gives ToT + GPT-4's accuracy and exact cost. ToT makes ~300 requests per puzzle, so on a free, rate-limited model even the one-puzzle smoke test can take a long time. Use this section only for a small same-model comparison (ToT vs. our method on the same model, a few puzzles).

Same algorithm and prompts as ToT (`src/crossword/tot.py`, checked state-for-state against ToT's original notebook), with every call counted.

**Choose the model here.** All free options:
* `MODEL_CHOICE = "gemini"`: Google's Gemini Flash on the free tier (needs the `GEMINI_API_KEY` secret). This is the GPT-4-class model. The free tier is rate-limited, so a full ToT run is slow (it makes ~300 requests per puzzle) and may stop at the daily limit. Re-run the cell the next day and it resumes.
* `MODEL_CHOICE = "qwen"`: Qwen2.5-7B-Instruct running on the Colab GPU, our small open model (Runtime → Change runtime type → a GPU). The first run downloads ~15 GB.
* `MODEL_CHOICE = "dry"`: a fake model that knows the answers. Checks the pipeline end to end; its scores mean nothing.
* `MODEL_CHOICE = "openai"`: paid OpenAI models, if we ever get credit.

Every response is cached in `cache/llm_cache.sqlite`, so re-running a cell replays the same answers for free.

In [22]:
MODEL_CHOICE = "dry"   # "gemini", "qwen", "dry" or "openai"
SPLIT = "test"

GEMINI_MODEL = "gemini-3.8-flash"            # if this errors, use a Flash model listed in AI Studio
QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
OPENAI_MODEL = "gpt-4o-mini"

import re
from crossword import SLOTS
from crossword.llm import make_llm, ScriptedLLM
from crossword.tot import ToTSolver, ToTConfig

def oracle_fn_for(puzzle):
    gold = dict(zip(SLOTS, puzzle.answers))
    def fn(prompt, n):
        if prompt.startswith("Evaluate if there exists"):
            return ["sure"] * n
        unfilled = re.findall(r"^([hv][1-5])\. .*: _____$", prompt, flags=re.M)
        return ["\n".join(f"{s}. {gold[s].lower()} (certain)" for s in unfilled)] * n
    return fn

if MODEL_CHOICE == "gemini":
    assert os.environ.get("GEMINI_API_KEY"), "add the GEMINI_API_KEY secret and re-run the setup cell"
    # newer Gemini models always 'think'; low effort keeps thinking from eating the token budget
    llm = make_llm("gemini", GEMINI_MODEL, request_kwargs={"reasoning_effort": "low"})
elif MODEL_CHOICE == "qwen":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=True)
    llm = make_llm("hf", QWEN_MODEL)
elif MODEL_CHOICE == "openai":
    llm = make_llm("openai", OPENAI_MODEL)
else:
    print("Dry run with a scripted model (scores are meaningless).")
    llm = ScriptedLLM(oracle_fn_for(get_split(SPLIT)[0]), model="scripted-oracle")
DRY_RUN = MODEL_CHOICE == "dry"
print("model:", llm.model)

Dry run with a scripted model (scores are meaningless).
model: scripted-oracle


### 3a · Smoke test: one puzzle, small search budget

Run this first. It shows real model output and projects the cost of the full run before you commit to it.

In [23]:
smoke_cfg = ToTConfig(time_limit=10)          # ToT uses 100; 10 keeps this quick
solver = ToTSolver(llm, smoke_cfg, verbose=True)
rec = solver.solve(get_split(SPLIT)[0])

print("\nfinal board:", rec["board"], " scores:", rec["scores"])
print("usage:", rec["usage"])

# rough projection: full run = ~100 nodes/puzzle x 20 puzzles
if rec["nodes"]:
    from crossword.llm import PRICES_PER_M
    scale = (100 / rec["nodes"]) * len(get_split(SPLIT))
    tok_in, tok_out = rec["usage"]["prompt_tokens"] * scale, rec["usage"]["completion_tokens"] * scale
    print(f"projected full {SPLIT} run: ~{tok_in/1e6:.1f}M input + ~{tok_out/1e6:.1f}M output tokens (upper bound)")
    if llm.model in PRICES_PER_M:
        pin, pout = PRICES_PER_M[llm.model]
        print(f"≈ ${(tok_in * pin + tok_out * pout) / 1e6:.2f} at list price for {llm.model} (check current pricing)")

0 ['h1. agend'] 0.1 {'sure': 1, 'maybe': 0, 'impossible': 0}
1 ['h1. agend', 'h2. motor'] 0.2 {'sure': 7, 'maybe': 0, 'impossible': 0}
2 ['h1. agend', 'h2. motor', 'h3. artsy'] 0.3 {'sure': 8, 'maybe': 0, 'impossible': 0}
3 ['h1. agend', 'h2. motor', 'h3. artsy', 'h4. salle'] 0.4 {'sure': 9, 'maybe': 0, 'impossible': 0}
4 ['h1. agend', 'h2. motor', 'h3. artsy', 'h4. salle', 'h5. sleer'] 1.0 {'sure': 10, 'maybe': 0, 'impossible': 0}
5 ['h1. agend', 'h2. motor', 'h3. artsy', 'h5. sleer'] 0.4 {'sure': 9, 'maybe': 0, 'impossible': 0}
6 ['h1. agend', 'h2. motor', 'h3. artsy', 'h5. sleer', 'h4. salle'] 1.0 {'sure': 10, 'maybe': 0, 'impossible': 0}
7 ['h1. agend', 'h2. motor', 'h4. salle'] 0.3 {'sure': 8, 'maybe': 0, 'impossible': 0}
8 ['h1. agend', 'h2. motor', 'h4. salle', 'h3. artsy'] 0.4 {'sure': 9, 'maybe': 0, 'impossible': 0}
9 ['h1. agend', 'h2. motor', 'h4. salle', 'h3. artsy', 'h5. sleer'] 1.0 {'sure': 10, 'maybe': 0, 'impossible': 0}

final board: AGENDMOTORARTSYSALLESLEER  scores: 

### 3b · Full baseline run

Set `RUN_FULL = True` to run all puzzles in the split with ToT's settings (time limit 100, prune, 3 children per state, 8 proposal samples).

Results are appended to `results/tot_dfs__<model>__<split>.jsonl` after **each** puzzle. If Colab disconnects, re-run this cell and it picks up where it stopped.

In [24]:
RUN_FULL = False

from crossword.experiment import run, results_path, load_results

out_path = results_path("tot_dfs", llm.model, SPLIT)
if RUN_FULL and not DRY_RUN:
    records = run(ToTSolver(llm, ToTConfig()), get_split(SPLIT), out_path)
else:
    records = load_results(out_path)
    print(f"RUN_FULL is off (or dry run); loaded {len(records)} saved records from {out_path.name}")
print("billed this session:", llm.usage.billed_calls, "calls,",
      llm.usage.billed_prompt_tokens + llm.usage.billed_completion_tokens, "tokens")

RUN_FULL is off (or dry run); loaded 0 saved records from tot_dfs__scripted-oracle__test.jsonl
billed this session: 43 calls, 22544 tokens


## 4 · Results so far

Every results file in `results/` (yours and your teammate's) next to the paper's GPT-4 numbers. Accuracy uses ToT's rule; `oracle_*` is the "+best state" upper bound. Cost columns are per puzzle and count cached calls too, i.e. what the method costs, not what this session paid.

In [25]:
from pathlib import Path
rows = [{"method": "ToT + GPT-4 (paper logs)", **summarize(tot_paper_records(prune=True))}]
for f in sorted(Path("results").glob("*.jsonl")):
    recs = load_results(f)
    if recs:
        rows.append({"method": f.stem, **summarize(recs)})
summary = pd.DataFrame(rows).set_index("method")
Path("results").mkdir(exist_ok=True)
summary.to_csv("results/summary.csv")
summary

,n,letter_%,word_%,game_%,games_solved,oracle_letter_%,oracle_word_%,oracle_game_%,oracle_games_solved
method,,,,,,,,,
ToT + GPT-4 (paper logs),20,78.0,60.0,20.0,4,82.4,67.5,35.0,7


## 5 · Share results (Colab)

Results only reach your teammate once they are pushed. Locally, commit `results/` as usual. On Colab, fill in your name and email and set `PUSH = True`. This needs a `GITHUB_TOKEN` secret with write access.

In [26]:
PUSH = False
GIT_NAME, GIT_EMAIL = "Your Name", "you@example.com"

if PUSH and IN_COLAB:
    for cmd in (["git", "config", "user.name", GIT_NAME], ["git", "config", "user.email", GIT_EMAIL],
                ["git", "pull", "-q", "--rebase", "--autostash"], ["git", "add", "results/"],
                ["git", "commit", "-q", "-m", f"Add ToT baseline results ({llm.model}, {SPLIT})"],
                ["git", "push", "-q"]):
        print(" ".join(cmd[:3]), "->", subprocess.run(cmd).returncode)